# Structured-output GP: HigherOrderGP and LatentKroneckerGP

This notebook compares two robotorchan models for structured outputs using a synthetic spectroscopy example.

- `HigherOrderGP`: treats the output as a tensor and exploits separable output structure.
- `LatentKroneckerGP`: models the product space between design variables `X` and an explicit output coordinate `T` such as wavelength, time, or position.


## 1. When to use these models

Use `HigherOrderGP` when each experiment returns an image, spectrum grid, or another tensor-valued response and the tensor shape itself is meaningful.

Use `LatentKroneckerGP` when the output is naturally indexed by a coordinate `T` (wavelength, time, position, etc.) and you want to query or interpolate along that axis explicitly.


In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll, fit_gpytorch_mll_torch
from linear_operator.settings import _fast_solves
from robotorchan.models import HigherOrderGP, LatentKroneckerGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 2. Synthetic spectroscopy data

`X` represents two process variables. Each row produces a spectrum observed on a wavelength axis `T`.


In [ ]:
n_train = 12
n_wavelengths = 32

train_X = torch.rand(n_train, 2)
train_T = torch.linspace(0.0, 1.0, n_wavelengths).unsqueeze(-1)

def spectrum(X, T):
    x1 = X[..., 0].unsqueeze(-1)
    x2 = X[..., 1].unsqueeze(-1)
    t = T.squeeze(-1)
    peak1 = torch.exp(-0.5 * ((t - (0.25 + 0.15 * x1)) / 0.07) ** 2)
    peak2 = 0.7 * torch.exp(-0.5 * ((t - (0.70 - 0.12 * x2)) / 0.10) ** 2)
    baseline = 0.15 * x1 + 0.08 * x2
    return peak1 + peak2 + baseline

train_Y = spectrum(train_X, train_T)
train_Y = train_Y + 0.02 * torch.randn_like(train_Y)

print(train_X.shape)
print(train_T.shape)
print(train_Y.shape)


In [ ]:
plt.figure(figsize=(8, 4))
for i in range(min(6, n_train)):
    plt.plot(train_T.squeeze(-1), train_Y[i], alpha=0.8)
plt.xlabel("normalized wavelength")
plt.ylabel("response")
plt.title("Synthetic training spectra")
plt.show()


## 3. HigherOrderGP

For HOGP, the response tensor is supplied directly as `train_Y`. A one-dimensional spectrum is still a structured output, but HOGP is especially useful for higher-order arrays such as images or spatial-temporal grids.

During fitting, BoTorch recommends enabling specialized fast solves and using `fit_gpytorch_mll_torch()`.


In [ ]:
hogp = HigherOrderGP(
    train_X=train_X,
    train_Y=train_Y,
)

print("supports_mll:", hogp.supports_mll)
print("raw_train_X:", hogp.raw_train_X.shape)
print("raw_train_Y:", hogp.raw_train_Y.shape)
print("raw_train_Yvar:", hogp.raw_train_Yvar)

hogp_mll = hogp.make_mll()
with _fast_solves(True):
    fit_gpytorch_mll_torch(hogp_mll, step_limit=75)

hogp.eval()


In [ ]:
test_X = torch.tensor([[0.65, 0.30]])
hogp_posterior = hogp.posterior(test_X)
hogp_mean = hogp_posterior.mean.squeeze(0).detach()

print("HOGP posterior mean shape:", hogp_mean.shape)


## 4. LatentKroneckerGP

LatentKroneckerGP receives the output coordinate explicitly as `train_T`. robotorchan therefore retains `raw_train_T` in addition to the common raw training tensors.

The model uses iterative methods for efficient fitting and posterior inference.


In [ ]:
lkgp = LatentKroneckerGP(
    train_X=train_X,
    train_T=train_T,
    train_Y=train_Y,
)

print("supports_mll:", lkgp.supports_mll)
print("raw_train_X:", lkgp.raw_train_X.shape)
print("raw_train_T:", lkgp.raw_train_T.shape)
print("raw_train_Y:", lkgp.raw_train_Y.shape)

lkgp_mll = lkgp.make_mll()
with lkgp.use_iterative_methods():
    fit_gpytorch_mll(lkgp_mll)

lkgp.eval()


In [ ]:
with lkgp.use_iterative_methods():
    lkgp_posterior = lkgp.posterior(test_X, train_T)

lkgp_mean = lkgp_posterior.mean.squeeze(0).detach()
lkgp_std = lkgp_posterior.variance.sqrt().squeeze(0).detach()

print("LatentKronecker posterior mean shape:", lkgp_mean.shape)


## 5. Compare predictions

The true spectrum is shown only because this is synthetic data. In a real application, the main comparison is how each model expresses structured-output assumptions and whether the explicit `T` axis is useful.


In [ ]:
true_y = spectrum(test_X, train_T).squeeze(0)

plt.figure(figsize=(9, 4))
plt.plot(train_T.squeeze(-1), true_y, label="true")
plt.plot(train_T.squeeze(-1), hogp_mean, label="HigherOrderGP")
plt.plot(train_T.squeeze(-1), lkgp_mean, label="LatentKroneckerGP")
plt.fill_between(
    train_T.squeeze(-1),
    lkgp_mean - 1.96 * lkgp_std,
    lkgp_mean + 1.96 * lkgp_std,
    alpha=0.2,
    label="LatentKronecker 95% interval",
)
plt.xlabel("normalized wavelength")
plt.ylabel("response")
plt.legend()
plt.title("Structured-output posterior at one process condition")
plt.show()


## 6. Querying a new T grid with LatentKroneckerGP

A major advantage of an explicit output coordinate is that the posterior can be evaluated on a new wavelength grid.


In [ ]:
test_T = torch.linspace(0.05, 0.95, 50).unsqueeze(-1)

with lkgp.use_iterative_methods():
    posterior_new_T = lkgp.posterior(test_X, test_T)

print("new T:", test_T.shape)
print("posterior mean:", posterior_new_T.mean.shape)


## 7. Model selection

| Situation | Recommended model |
|---|---|
| Tensor-valued output, e.g. image or multidimensional grid | `HigherOrderGP` |
| Spectrum / time series / spatial profile with meaningful coordinate | `LatentKroneckerGP` |
| Need to evaluate at new output-coordinate locations | `LatentKroneckerGP` |
| Output tensor has multiple structured axes | `HigherOrderGP` |

For spectra with a physically meaningful wavelength axis, `LatentKroneckerGP` is often the more interpretable choice because wavelength is represented explicitly as `T`.

For images or genuinely higher-order outputs, `HigherOrderGP` is the more natural representation.


## 8. Notes for Bayesian optimization

Both models can be used as surrogates for structured outputs, but BO usually needs a scalar or low-dimensional objective derived from the structured response, for example:

- peak intensity,
- integrated spectral area,
- distance to a target spectrum,
- maximum response over wavelength,
- a physics-informed scalar score.

Define that objective explicitly before choosing an acquisition function.
